# 04 · Run tracks

Full pipeline (notebooks 00–03) for every bias tolerance in `params.BIAS_TOLERANCES`, for one or many tracks. Results are merged into six shared CSV tables in the output folder (locked writes, so parallel runs are safe; re-running a track replaces its rows).

For long batches the command line is more convenient:

```bash
is2retreat --config configs/north_slope.toml --tracks-file configs/tracks_filtered.txt
```

Requires the package installed from the repository root (`pip install -e .`).
Paths come from `configs/north_slope.toml`; all parameters and their defaults are in `src/is2retreat/config.py`.

In [ ]:
TRACKS = ["0129"]        # or None to read TRACKS_FILE
TRACKS_FILE = "../configs/tracks_filtered.txt"
CONFIG = "../configs/north_slope.toml"
SOURCE = "auto"
OUTDIR = None            # None = [paths] outdir from the config

In [ ]:
from dataclasses import replace
from pathlib import Path
import time

import matplotlib.pyplot as plt
import pandas as pd

from is2retreat import TrackSkipped, load_config, run_track
from is2retreat.outputs import OutputFiles

paths, params = load_config(CONFIG)
if OUTDIR is not None:
    paths = replace(paths, outdir=Path(OUTDIR).resolve())

if TRACKS is None:
    TRACKS = [t.strip() for t in Path(TRACKS_FILE).read_text().splitlines() if t.strip()]

print(f"{len(TRACKS)} track(s) -> {paths.outdir}")
print("Bias tolerances:", params.BIAS_TOLERANCES)

In [ ]:
status = []
for track_id in TRACKS:
    start = time.time()
    try:
        result = run_track(track_id, paths, params, source=SOURCE, verbose=False)
        n_gie = 0 if result.gie is None else len(result.gie.summary_df)
        status.append({"track_id": track_id, "status": "ok", "dsas_clusters": len(result.dsas_summary),
                       "gie_clusters": n_gie, "note": result.gie_skip_reason, "seconds": round(time.time() - start)})
    except TrackSkipped as e:
        status.append({"track_id": track_id, "status": "skipped", "note": str(e).splitlines()[0]})
    except Exception as e:
        status.append({"track_id": track_id, "status": "failed", "note": repr(e)})

status = pd.DataFrame(status)
status

## Output tables

In [ ]:
files = OutputFiles.in_dir(paths.outdir, params.RES_TAG)
gie = pd.read_csv(files.gie_summary, dtype={"track_id": str})
gie = gie[gie["track_id"].isin([str(t).zfill(4) for t in TRACKS])]
gie[["track_id", "bias_tolerance", "gt_family", "cluster_id", "n_beams",
     "NSM_measured", "NSM_corrected", "EPR_measured", "EPR_corrected"]].head(20)

## Sensitivity to the bias tolerance

In [ ]:
by_tol = gie.groupby("bias_tolerance").agg(
    clusters=("cluster_id", "size"),
    median_EPR_measured=("EPR_measured", "median"),
    median_EPR_corrected=("EPR_corrected", "median"),
)

fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot(by_tol.index, by_tol["median_EPR_measured"], "o-", label="median EPR measured")
ax1.plot(by_tol.index, by_tol["median_EPR_corrected"], "s-", label="median EPR GIE corrected")
ax1.set_xlabel("bias tolerance (m)")
ax1.set_ylabel("EPR (m/yr)")
ax2 = ax1.twinx()
ax2.bar(by_tol.index, by_tol["clusters"], width=0.04, alpha=0.2, color="gray")
ax2.set_ylabel("clusters")
ax1.legend(loc="lower left", fontsize=8)
plt.show()
by_tol